# Supply Chain Risk & Supplier Analytics

This project reviews supplier delivery, lead time, and quality using fictional data.

## 1. Load the data

Upload the five CSV files to Colab, then run the code.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

supplier_master = pd.read_csv("/content/Supplier_Master.csv")
material_master = pd.read_csv("/content/Material_Master.csv")
purchase_orders = pd.read_csv("/content/Purchase_Orders.csv")
goods_receipts = pd.read_csv("/content/Goods_Receipts.csv")
quality_inspections = pd.read_csv("/content/Quality_Inspections.csv")

print("Supplier master:", supplier_master.shape)
print("Material master:", material_master.shape)
print("Purchase orders:", purchase_orders.shape)
print("Goods receipts:", goods_receipts.shape)
print("Quality inspections:", quality_inspections.shape)

display(purchase_orders.head())

## 2. Check the data

Check missing values, duplicate IDs, links between files, and date formats.

In [ ]:
print("Missing values in each file:")
print("Supplier master:", supplier_master.isnull().sum().sum())
print("Material master:", material_master.isnull().sum().sum())
print("Purchase orders:", purchase_orders.isnull().sum().sum())
print("Goods receipts:", goods_receipts.isnull().sum().sum())
print("Quality inspections:", quality_inspections.isnull().sum().sum())

print("\nDuplicate purchase order line IDs:", purchase_orders["PO_Line_ID"].duplicated().sum())
print("Duplicate goods receipt IDs:", goods_receipts["Goods_Receipt_ID"].duplicated().sum())
print("Duplicate inspection IDs:", quality_inspections["Inspection_ID"].duplicated().sum())

print("\nSupplier IDs connect:", purchase_orders["Supplier_ID"].isin(supplier_master["Supplier_ID"]).all())
print("Material IDs connect:", purchase_orders["Material_ID"].isin(material_master["Material_ID"]).all())
print("Receipt purchase order IDs connect:", goods_receipts["PO_Line_ID"].isin(purchase_orders["PO_Line_ID"]).all())
print("Inspection receipt IDs connect:", quality_inspections["Goods_Receipt_ID"].isin(goods_receipts["Goods_Receipt_ID"]).all())

purchase_orders["Order_Date"] = pd.to_datetime(purchase_orders["Order_Date"], errors="coerce")
purchase_orders["Promised_Delivery_Date"] = pd.to_datetime(purchase_orders["Promised_Delivery_Date"], errors="coerce")
goods_receipts["Receipt_Date"] = pd.to_datetime(goods_receipts["Receipt_Date"], errors="coerce")
quality_inspections["Inspection_Date"] = pd.to_datetime(quality_inspections["Inspection_Date"], errors="coerce")

print("\nPurchase order date range:")
print(purchase_orders["Order_Date"].min(), "to", purchase_orders["Order_Date"].max())

## 3. Review delivery performance

Combine receipts by purchase order line and compare the last receipt date with the promised date.

In [ ]:
receipt_summary = goods_receipts.groupby("PO_Line_ID").agg(
    Total_Received_Quantity=("Received_Quantity", "sum"),
    Final_Receipt_Date=("Receipt_Date", "max"),
    Number_of_Receipts=("Goods_Receipt_ID", "count")
).reset_index()

delivery_data = purchase_orders.merge(
    receipt_summary,
    on="PO_Line_ID",
    how="left"
)

delivery_data["Delivery_Delay_Days"] = (
    delivery_data["Final_Receipt_Date"] - delivery_data["Promised_Delivery_Date"]
).dt.days

delivery_data["Delivery_Status"] = "Not Yet Received"

delivery_data.loc[
    delivery_data["Final_Receipt_Date"].notna()
    & (delivery_data["Delivery_Delay_Days"] <= 0),
    "Delivery_Status"
] = "On Time"

delivery_data.loc[
    delivery_data["Final_Receipt_Date"].notna()
    & (delivery_data["Delivery_Delay_Days"] > 0),
    "Delivery_Status"
] = "Late"

print("Delivery status:")
print(delivery_data["Delivery_Status"].value_counts())

display(delivery_data[
    ["PO_Line_ID", "Promised_Delivery_Date", "Final_Receipt_Date",
     "Delivery_Delay_Days", "Delivery_Status"]
].head(10))

## 4. Compare supplier delivery

Compare on-time delivery for completed lines. Keep lines not yet received separate.

In [ ]:
delivery_data["On_Time_Flag"] = (delivery_data["Delivery_Status"] == "On Time").astype(int)
delivery_data["Late_Flag"] = (delivery_data["Delivery_Status"] == "Late").astype(int)
delivery_data["Not_Received_Flag"] = (delivery_data["Delivery_Status"] == "Not Yet Received").astype(int)

supplier_delivery = delivery_data.groupby("Supplier_ID").agg(
    Total_PO_Lines=("PO_Line_ID", "count"),
    On_Time_Lines=("On_Time_Flag", "sum"),
    Late_Lines=("Late_Flag", "sum"),
    Not_Received_Lines=("Not_Received_Flag", "sum")
).reset_index()

supplier_delivery["On_Time_Delivery_Percentage"] = (
    supplier_delivery["On_Time_Lines"]
    / (supplier_delivery["On_Time_Lines"] + supplier_delivery["Late_Lines"])
    * 100
).round(2)

supplier_delivery = supplier_delivery.merge(
    supplier_master[["Supplier_ID", "Supplier_Name"]],
    on="Supplier_ID",
    how="left"
)

print("Suppliers with purchase orders:", supplier_delivery["Supplier_ID"].nunique())
print("\n10 suppliers with the lowest on-time delivery percentage:")
display(supplier_delivery.sort_values("On_Time_Delivery_Percentage")[
    ["Supplier_Name", "Total_PO_Lines", "On_Time_Lines", "Late_Lines",
     "Not_Received_Lines", "On_Time_Delivery_Percentage"]
].head(10))

## 5. Review lead times

Calculate actual lead time for received lines and review the average and variation.

In [ ]:
delivery_data["Actual_Lead_Time_Days"] = (
    delivery_data["Final_Receipt_Date"] - delivery_data["Order_Date"]
).dt.days

completed_deliveries = delivery_data[
    delivery_data["Final_Receipt_Date"].notna()
].copy()

supplier_lead_time = completed_deliveries.groupby("Supplier_ID").agg(
    Completed_Lines=("PO_Line_ID", "count"),
    Average_Lead_Time_Days=("Actual_Lead_Time_Days", "mean"),
    Lead_Time_Std_Dev=("Actual_Lead_Time_Days", "std"),
    Minimum_Lead_Time_Days=("Actual_Lead_Time_Days", "min"),
    Maximum_Lead_Time_Days=("Actual_Lead_Time_Days", "max")
).reset_index()

supplier_lead_time[
    ["Average_Lead_Time_Days", "Lead_Time_Std_Dev",
     "Minimum_Lead_Time_Days", "Maximum_Lead_Time_Days"]
] = supplier_lead_time[
    ["Average_Lead_Time_Days", "Lead_Time_Std_Dev",
     "Minimum_Lead_Time_Days", "Maximum_Lead_Time_Days"]
].round(2)

supplier_lead_time = supplier_lead_time.merge(
    supplier_master[["Supplier_ID", "Supplier_Name"]],
    on="Supplier_ID",
    how="left"
)

print("10 suppliers with the longest average lead times:")
display(supplier_lead_time.sort_values(
    "Average_Lead_Time_Days", ascending=False
)[
    ["Supplier_Name", "Completed_Lines", "Average_Lead_Time_Days",
     "Lead_Time_Std_Dev", "Minimum_Lead_Time_Days", "Maximum_Lead_Time_Days"]
].head(10))

## 6. Review quality

Calculate the rejection rate using rejected quantity and inspected quantity.

In [ ]:
quality_data = quality_inspections.merge(
    goods_receipts[["Goods_Receipt_ID", "PO_Line_ID"]],
    on="Goods_Receipt_ID",
    how="left"
)

quality_data = quality_data.merge(
    purchase_orders[["PO_Line_ID", "Supplier_ID"]],
    on="PO_Line_ID",
    how="left"
)

supplier_quality = quality_data.groupby("Supplier_ID").agg(
    Inspection_Records=("Inspection_ID", "count"),
    Total_Inspected_Quantity=("Inspected_Quantity", "sum"),
    Total_Rejected_Quantity=("Rejected_Quantity", "sum")
).reset_index()

supplier_quality["Rejection_Rate_Percentage"] = (
    supplier_quality["Total_Rejected_Quantity"]
    / supplier_quality["Total_Inspected_Quantity"]
    * 100
).round(2)

supplier_quality = supplier_quality.merge(
    supplier_master[["Supplier_ID", "Supplier_Name"]],
    on="Supplier_ID",
    how="left"
)

print("10 suppliers with the highest rejection rates:")
display(supplier_quality.sort_values(
    "Rejection_Rate_Percentage", ascending=False
)[
    ["Supplier_Name", "Inspection_Records", "Total_Inspected_Quantity",
     "Total_Rejected_Quantity", "Rejection_Rate_Percentage"]
].head(10))

## 7. Combine supplier results

Combine delivery, lead-time, and quality results, then compare them with supplier targets.

In [ ]:
supplier_risk_summary = supplier_master[
    ["Supplier_ID", "Supplier_Name", "Supplier_Status",
     "Standard_Lead_Time_Days", "Historical_On_Time_Target", "Expected_Defect_Rate"]
].copy()

supplier_risk_summary = supplier_risk_summary.merge(
    supplier_delivery[
        ["Supplier_ID", "Total_PO_Lines", "On_Time_Lines", "Late_Lines",
         "Not_Received_Lines", "On_Time_Delivery_Percentage"]
    ],
    on="Supplier_ID",
    how="left"
)

supplier_risk_summary = supplier_risk_summary.merge(
    supplier_lead_time[
        ["Supplier_ID", "Completed_Lines", "Average_Lead_Time_Days",
         "Lead_Time_Std_Dev", "Minimum_Lead_Time_Days", "Maximum_Lead_Time_Days"]
    ],
    on="Supplier_ID",
    how="left"
)

supplier_risk_summary = supplier_risk_summary.merge(
    supplier_quality[
        ["Supplier_ID", "Inspection_Records", "Total_Inspected_Quantity",
         "Total_Rejected_Quantity", "Rejection_Rate_Percentage"]
    ],
    on="Supplier_ID",
    how="left"
)

supplier_risk_summary["Expected_On_Time_Target_Pct"] = (
    supplier_risk_summary["Historical_On_Time_Target"] * 100
)

supplier_risk_summary["Expected_Defect_Rate_Pct"] = (
    supplier_risk_summary["Expected_Defect_Rate"] * 100
)

supplier_risk_summary["Lead_Time_Above_Standard_Days"] = (
    supplier_risk_summary["Average_Lead_Time_Days"]
    - supplier_risk_summary["Standard_Lead_Time_Days"]
).round(2)

supplier_risk_summary["On_Time_Target_Gap"] = (
    supplier_risk_summary["On_Time_Delivery_Percentage"]
    - supplier_risk_summary["Expected_On_Time_Target_Pct"]
).round(2)

supplier_risk_summary["Quality_Target_Gap"] = (
    supplier_risk_summary["Rejection_Rate_Percentage"]
    - supplier_risk_summary["Expected_Defect_Rate_Pct"]
).round(2)

print("Supplier risk summary:", supplier_risk_summary.shape)
display(supplier_risk_summary[
    ["Supplier_Name", "Total_PO_Lines", "On_Time_Delivery_Percentage",
     "Average_Lead_Time_Days", "Rejection_Rate_Percentage",
     "On_Time_Target_Gap", "Lead_Time_Above_Standard_Days", "Quality_Target_Gap"]
].head(10))

## 8. Assign risk categories

Use delivery, lead-time, and quality warning signs to create a simple risk score. Flag suppliers with limited delivery history.

In [ ]:
supplier_risk_summary["Delivery_Risk_Flag"] = (
    supplier_risk_summary["On_Time_Target_Gap"] < 0
)

supplier_risk_summary["Lead_Time_Risk_Flag"] = (
    supplier_risk_summary["Lead_Time_Above_Standard_Days"] > 0
)

supplier_risk_summary["Quality_Risk_Flag"] = (
    supplier_risk_summary["Quality_Target_Gap"] > 0
)

supplier_risk_summary["Risk_Score"] = (
    supplier_risk_summary["Delivery_Risk_Flag"].astype(int)
    + supplier_risk_summary["Lead_Time_Risk_Flag"].astype(int)
    + supplier_risk_summary["Quality_Risk_Flag"].astype(int)
)

supplier_risk_summary.loc[
    supplier_risk_summary["Total_PO_Lines"].isna(),
    "Risk_Score"
] = float("nan")

supplier_risk_summary["Risk_Category"] = "Low"
supplier_risk_summary.loc[supplier_risk_summary["Risk_Score"] == 2, "Risk_Category"] = "Medium"
supplier_risk_summary.loc[supplier_risk_summary["Risk_Score"] == 3, "Risk_Category"] = "High"
supplier_risk_summary.loc[supplier_risk_summary["Risk_Score"].isna(), "Risk_Category"] = "Not Assessed"

supplier_risk_summary["Sample_Size_Flag"] = "Sufficient Sample"
supplier_risk_summary.loc[
    supplier_risk_summary["Completed_Lines"] < 30,
    "Sample_Size_Flag"
] = "Review Small Sample"
supplier_risk_summary.loc[
    supplier_risk_summary["Total_PO_Lines"].isna(),
    "Sample_Size_Flag"
] = "No Purchase Orders"
supplier_risk_summary.loc[
    supplier_risk_summary["Total_PO_Lines"].notna()
    & supplier_risk_summary["Completed_Lines"].isna(),
    "Sample_Size_Flag"
] = "No Completed Deliveries"

supplier_risk_summary["Review_Priority"] = "Routine Monitoring"
supplier_risk_summary.loc[
    supplier_risk_summary["Risk_Category"] == "Not Assessed",
    "Review_Priority"
] = "No Delivery Data"
supplier_risk_summary.loc[
    supplier_risk_summary["Sample_Size_Flag"] == "Review Small Sample",
    "Review_Priority"
] = "Validate Small Sample"
supplier_risk_summary.loc[
    (supplier_risk_summary["Risk_Category"] == "High")
    & (supplier_risk_summary["Sample_Size_Flag"] == "Sufficient Sample"),
    "Review_Priority"
] = "Priority Review"
supplier_risk_summary.loc[
    (supplier_risk_summary["Risk_Category"] == "Medium")
    & (supplier_risk_summary["Sample_Size_Flag"] == "Sufficient Sample"),
    "Review_Priority"
] = "Monitor Closely"

print("Risk categories:")
print(supplier_risk_summary["Risk_Category"].value_counts())

print("\nReview priorities:")
print(supplier_risk_summary["Review_Priority"].value_counts())

print("\nSuppliers requiring priority review:")
display(supplier_risk_summary[
    supplier_risk_summary["Review_Priority"] == "Priority Review"
][
    ["Supplier_Name", "Completed_Lines", "On_Time_Delivery_Percentage",
     "Lead_Time_Above_Standard_Days", "Quality_Target_Gap", "Risk_Score"]
].sort_values("Risk_Score", ascending=False))

## 9. View the results

Use the charts to compare supplier risk, delivery, lead time, and quality.

In [ ]:
risk_counts = supplier_risk_summary["Risk_Category"].value_counts()

plt.figure(figsize=(8, 5))
risk_counts.plot(kind="bar")
plt.title("Supplier Risk Categories")
plt.xlabel("Risk Category")
plt.ylabel("Number of Suppliers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
delivery_chart = supplier_risk_summary.dropna(
    subset=["On_Time_Delivery_Percentage"]
).sort_values("On_Time_Delivery_Percentage").head(10)

plt.figure(figsize=(9, 5))
plt.barh(
    delivery_chart["Supplier_Name"],
    delivery_chart["On_Time_Delivery_Percentage"]
)
plt.title("10 Suppliers with the Lowest On-Time Delivery")
plt.xlabel("On-Time Delivery (%)")
plt.ylabel("Supplier")
plt.tight_layout()
plt.show()

In [ ]:
priority_suppliers = supplier_risk_summary[
    supplier_risk_summary["Review_Priority"] == "Priority Review"
].sort_values("Lead_Time_Above_Standard_Days")

plt.figure(figsize=(9, 5))
plt.barh(
    priority_suppliers["Supplier_Name"],
    priority_suppliers["Lead_Time_Above_Standard_Days"]
)
plt.title("Lead Time Above Standard for Priority-Review Suppliers")
plt.xlabel("Days Above Standard")
plt.ylabel("Supplier")
plt.tight_layout()
plt.show()

In [ ]:
quality_chart = supplier_risk_summary.dropna(
    subset=["Quality_Target_Gap"]
).sort_values("Quality_Target_Gap")

plt.figure(figsize=(9, 6))
plt.barh(
    quality_chart["Supplier_Name"],
    quality_chart["Quality_Target_Gap"]
)
plt.axvline(x=0, linestyle="--")
plt.title("Actual Rejection Rate Compared with Expected Rate")
plt.xlabel("Gap (Percentage Points)")
plt.ylabel("Supplier")
plt.tight_layout()
plt.show()

## 10. Key figures

These figures summarize the supplier results. The averages are not weighted by order volume.

In [ ]:
priority_count = (
    supplier_risk_summary["Review_Priority"] == "Priority Review"
).sum()

high_risk_count = (
    supplier_risk_summary["Risk_Category"] == "High"
).sum()

not_assessed_count = (
    supplier_risk_summary["Risk_Category"] == "Not Assessed"
).sum()

average_on_time = supplier_risk_summary["On_Time_Delivery_Percentage"].mean()
average_rejection = supplier_risk_summary["Rejection_Rate_Percentage"].mean()

priority_avg_lead_gap = priority_suppliers["Lead_Time_Above_Standard_Days"].mean()

print("Total suppliers:", len(supplier_risk_summary))
print("High-risk suppliers:", high_risk_count)
print("Suppliers requiring priority review:", priority_count)
print("Suppliers not assessed:", not_assessed_count)
print("Average supplier on-time delivery:", round(average_on_time, 2), "%")
print("Average supplier rejection rate:", round(average_rejection, 2), "%")
print("Average lead-time gap for priority-review suppliers:", round(priority_avg_lead_gap, 2), "days")

## 11. Findings and actions

### Findings

- 9 suppliers are classified as high risk, 7 as medium risk, and 2 as low risk.
- 12 suppliers have no purchase orders in the dataset, so their delivery risk could not be assessed.
- 8 suppliers are selected for priority review because they have a high-risk score and at least 30 completed delivery lines.
- Average supplier-level on-time delivery is about 48.07% among suppliers with delivery data.
- Suppliers selected for priority review exceed their standard lead time by about 2.72 days on average.
- Some suppliers have rejection rates above their expected defect rates. The size of the gap should be considered alongside the overall rejection rate.

### Suggested actions

- Start with high-risk suppliers that have enough delivery history to support a meaningful review.
- Check the reasons for late deliveries and lead-time overruns before changing purchasing plans.
- Discuss corrective actions with suppliers whose rejection rates are above their expected levels.
- Collect more purchasing and delivery evidence for suppliers with no activity or small samples.
- Review the indicators again when new purchase orders, receipts, and inspections become available.

### Limitations

The data is synthetic, and the risk score is a simple rule-based screening method. The results can help organize a supplier review, but they should not be used to make automatic procurement decisions. Small samples can also make supplier percentages less reliable.